# Energy Data Windowing
Exploratory analysis of the NESO historic generation mix dataset using DuckDB window functions.

In [10]:
import duckdb
import polars as pl                     
pl.Config.set_tbl_rows(200)


con = duckdb.connect()
con.execute("CREATE VIEW generation AS SELECT * FROM read_csv_auto('data/df_fuel_ckan.csv')")

In [2]:
# How many rows and what date range
con.sql("SELECT COUNT(*) AS rows, MIN(DATETIME) as from_date, MAX(DATETIME) as to_date FROM generation").show()

┌────────┬─────────────────────┬─────────────────────┐
│  rows  │      from_date      │       to_date       │
│ int64  │      timestamp      │      timestamp      │
├────────┼─────────────────────┼─────────────────────┤
│ 301142 │ 2009-01-01 00:00:00 │ 2026-03-06 18:30:00 │
└────────┴─────────────────────┴─────────────────────┘



In [3]:
# What columns do we have
con.sql("DESCRIBE generation").show()

┌──────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│   column_name    │ column_type │  null   │   key   │ default │  extra  │
│     varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├──────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ DATETIME         │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ GAS              │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ COAL             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ NUCLEAR          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ WIND             │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ WIND_EMB         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ HYDRO            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ IMPORTS          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ BIOMASS          │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ OTHER            │ DOUB

In [4]:
# Preview the first few rows
con.sql("SELECT * FROM generation LIMIT 5").show()

┌─────────────────────┬────────┬─────────┬─────────┬────────┬──────────┬───────┬─────────┬─────────┬────────┬────────┬─────────┬────────────┬──────────────────┬────────────┬─────────────┬───────────┬─────────┬──────────┬───────────┬──────────────┬───────────┬───────────────┬────────────┬──────────────┬──────────────┬────────────┬────────────┬──────────────┬─────────────────┬─────────────────┬──────────────────┬────────────────┬─────────────┐
│      DATETIME       │  GAS   │  COAL   │ NUCLEAR │  WIND  │ WIND_EMB │ HYDRO │ IMPORTS │ BIOMASS │ OTHER  │ SOLAR  │ STORAGE │ GENERATION │ CARBON_INTENSITY │ LOW_CARBON │ ZERO_CARBON │ RENEWABLE │ FOSSIL  │ GAS_perc │ COAL_perc │ NUCLEAR_perc │ WIND_perc │ WIND_EMB_perc │ HYDRO_perc │ IMPORTS_perc │ BIOMASS_perc │ OTHER_perc │ SOLAR_perc │ STORAGE_perc │ GENERATION_perc │ LOW_CARBON_perc │ ZERO_CARBON_perc │ RENEWABLE_perc │ FOSSIL_perc │
│      timestamp      │ double │ double  │ double  │ double │  double  │ int64 │ double  │ double  │ double 

In [5]:
# Get a 24hr rolling average for each fuel type

con.sql("""SELECT DATETIME, GAS, WIND, SOLAR, COAL, NUCLEAR,
        AVG(GAS) OVER (ORDER BY DATETIME ROWS BETWEEN 47 PRECEDING AND CURRENT ROW) as gas_24hr_avg,
        AVG(WIND) OVER (ORDER BY DATETIME ROWS BETWEEN 47 PRECEDING AND CURRENT ROW) as wind_24hr_avg,
        AVG(COAL) OVER (ORDER BY DATETIME ROWS BETWEEN 47 PRECEDING AND CURRENT ROW) as coal_24hr_avg,
        AVG(SOLAR) OVER (ORDER BY DATETIME ROWS BETWEEN 47 PRECEDING AND CURRENT ROW) as solar_24hr_avg,
        AVG(NUCLEAR) OVER (ORDER BY DATETIME ROWS BETWEEN 47 PRECEDING AND CURRENT ROW) as nuclear_24hr_avg
        from generation""").show()

┌─────────────────────┬─────────┬────────┬────────┬─────────┬─────────┬────────────────────┬────────────────────┬────────────────────┬────────────────┬───────────────────┐
│      DATETIME       │   GAS   │  WIND  │ SOLAR  │  COAL   │ NUCLEAR │    gas_24hr_avg    │   wind_24hr_avg    │   coal_24hr_avg    │ solar_24hr_avg │ nuclear_24hr_avg  │
│      timestamp      │ double  │ double │ double │ double  │ double  │       double       │       double       │       double       │     double     │      double       │
├─────────────────────┼─────────┼────────┼────────┼─────────┼─────────┼────────────────────┼────────────────────┼────────────────────┼────────────────┼───────────────────┤
│ 2009-01-01 00:00:00 │  8367.0 │  248.0 │    0.0 │ 15037.0 │  7099.0 │             8367.0 │              248.0 │            15037.0 │            0.0 │            7099.0 │
│ 2009-01-01 00:30:00 │  8495.0 │  229.0 │    0.0 │ 15095.0 │  7088.0 │             8431.0 │              238.5 │            15066.0 │      

# Observations on the data so far

- Row 1 — the average equals the raw value, because there's only 1 row in the
  window so far. It builds up correctly as more rows accumulate.
- The window is working and by row 10 (04:30) you can see the gas average
  smoothing out the raw values                                                  
- Solar is 0.0 throughout but the dates show it is January 2009 and night time; it may also be that solar barely existed in the UK grid at that time. Data exploration at later dates and times might show different patterns for solar. 

In [6]:
# How the fuel mix has changed over the years

con.sql("""SELECT
        YEAR(DATETIME) as year,
        ROUND(AVG(COAL_perc), 1) AS coal_pct,
        ROUND(AVG(GAS_perc), 1) AS gas_pct,
        ROUND(AVG(NUCLEAR_perc), 1) AS nuclear_pct,
        ROUND(AVG(WIND_perc), 1) AS wind_pct,
        ROUND(AVG(SOLAR_perc), 1) AS solar_pct,
        ROUND(AVG(BIOMASS_perc), 1) AS biomass_pct,
        ROUND(AVG(FOSSIL_perc), 1) AS fossil_pct
        from generation
        GROUP BY year
        ORDER BY year""").show()

┌───────┬──────────┬─────────┬─────────────┬──────────┬───────────┬─────────────┬────────────┐
│ year  │ coal_pct │ gas_pct │ nuclear_pct │ wind_pct │ solar_pct │ biomass_pct │ fossil_pct │
│ int64 │  double  │ double  │   double    │  double  │  double   │   double    │   double   │
├───────┼──────────┼─────────┼─────────────┼──────────┼───────────┼─────────────┼────────────┤
│  2009 │     28.1 │    45.7 │        20.6 │      1.0 │       0.0 │         0.0 │       73.8 │
│  2010 │     29.0 │    47.7 │        18.0 │      1.1 │       0.0 │         0.0 │       76.8 │
│  2011 │     30.7 │    39.8 │        20.9 │      3.1 │       0.0 │         0.0 │       70.5 │
│  2012 │     42.0 │    25.0 │        21.2 │      4.0 │       0.0 │         0.0 │       67.0 │
│  2013 │     39.1 │    23.6 │        21.3 │      5.9 │       0.6 │         0.0 │       62.7 │
│  2014 │     30.8 │    27.3 │        20.1 │      6.9 │       1.2 │         0.0 │       58.1 │
│  2015 │     23.8 │    26.9 │        22.4 │      

# Observations on the data so far

- Solar is 0% until 2013 when it becomes 0.6%, its highest year was 2025 at 6.1%
- Biomas is 0% until 2017 when it becomes 0.7%, its highest year was 2025 at 7.0%. Jumps in suddenly around 2017-2018 — that's Drax power station converting
  from coal
- Coal is solidly around 30-40% from 2009 to 2015 when it suddenly collapses to 8.5% in 2016 and slides further down to 0% in 2025
- Overall fossil fuel mix is the 70s 2009-2011 but has come down to high 20s in recent years. 
- 2026 data is only partial, covering just January, so solar is still low and wind is higher, which reflects winter patterns.

In [11]:
# Ranking fuel types per time period



con.sql("""WITH unpivoted AS (
        SELECT YEAR(DATETIME) as year, fuel_type, mw
        FROM (
        UNPIVOT generation
        ON GAS, WIND, SOLAR, COAL, NUCLEAR, BIOMASS
        INTO NAME fuel_type VALUE mw
        )
        ),
        ranked AS (
            SELECT year, fuel_type, ROUND(AVG(mw), 1) AS avg_mw,
            RANK() OVER (PARTITION BY year ORDER BY AVG(mw) DESC) AS rank
            FROM unpivoted
            GROUP BY year, fuel_type
        )
        SELECT * FROM ranked
        ORDER BY year, rank
        """).pl()

year,fuel_type,avg_mw,rank
i64,str,f64,i64
2009,"""GAS""",16825.5,1
2009,"""COAL""",11278.0,2
2009,"""NUCLEAR""",7420.3,3
2009,"""WIND""",379.7,4
2009,"""SOLAR""",0.0,5
2009,"""BIOMASS""",0.0,5
2010,"""GAS""",17931.4,1
2010,"""COAL""",11739.8,2
2010,"""NUCLEAR""",6679.3,3


## Observations on the data so far

- In the early years of this data, coal is number 2, gas is number 1 but in recent years coal is 6th place and so far in 2026, wind is 2nd place.
- Gas is consistently number 1 except in 2012-2014 coal was actually rank 1, briefly overtaking gas
- Since 2020 wind has been consistently number 2 and it is only slightly behind gas in 2026 (a gap of 268 MW)

In [15]:
# Using LAG to detect ramp up and down events in the data

con.sql("""SELECT DATETIME,
        GAS - LAG(GAS) OVER (ORDER BY DATETIME) as gas_delta,
        WIND - LAG(WIND) OVER (ORDER BY DATETIME) as wind_delta,
        COAL - LAG(COAL) OVER (ORDER BY DATETIME) as coal_delta,
        SOLAR - LAG(SOLAR) OVER (ORDER BY DATETIME) as solar_delta,
        NUCLEAR - LAG(NUCLEAR) OVER (ORDER BY DATETIME) as nuclear_delta,        
        from generation
        ORDER BY DATETIME
        """).show()

┌─────────────────────┬───────────┬────────────┬────────────┬─────────────┬───────────────┐
│      DATETIME       │ gas_delta │ wind_delta │ coal_delta │ solar_delta │ nuclear_delta │
│      timestamp      │  double   │   double   │   double   │   double    │    double     │
├─────────────────────┼───────────┼────────────┼────────────┼─────────────┼───────────────┤
│ 2009-01-01 00:00:00 │      NULL │       NULL │       NULL │        NULL │          NULL │
│ 2009-01-01 00:30:00 │     128.0 │      -19.0 │       58.0 │         0.0 │         -11.0 │
│ 2009-01-01 01:00:00 │     -24.0 │      -22.0 │       -7.0 │         0.0 │         -14.0 │
│ 2009-01-01 01:30:00 │    -153.0 │      -16.0 │      -53.0 │         0.0 │         -10.0 │
│ 2009-01-01 02:00:00 │     -23.0 │      -16.0 │      -30.0 │         0.0 │         -12.0 │
│ 2009-01-01 02:30:00 │       9.0 │      -13.0 │       13.0 │         0.0 │         -11.0 │
│ 2009-01-01 03:00:00 │      54.0 │       -7.0 │       -8.0 │         0.0 │     